# Task 2: TFIM Phase Classification, Standalone Notebook

This notebook runs the paper-aligned **Task 2** study directly inside the notebook.

It matches the manuscript design at the experiment-logic level:

- TFIM ground states with `n = 6`
- `g/J` values on `[0.2, 1.8]` with 8 ferromagnetic and 8 paramagnetic settings
- `200` Z-basis sequences per coupling, `3200` total samples
- `70/30` split with a per-coupling stratified `50%` low-data regime (`1120` train)
- fixed `T = 50` reservoir rollout with `p_dep = 0.01`
- baselines: `Linear`, `Ridge`, `MLP`, `MLP-S`, `Ridge-PCA-d3`, `Tensor-Ridge`, `ML-QRC-M`
- Task 2 `d`-sweep: `MLP (ref.)`, `MLP-S (ref.)`, `ML-QRC-S/M/L/XL`

The only cache reused across runs is the TFIM QRC feature tensor bank written under
`notebooks/artifacts/task3_tfim/`.


In [ ]:
%pip install -q numpy pandas matplotlib torch scikit-learn pennylane jupyter


In [ ]:
from __future__ import annotations

import os
import random
import time
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def repo_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "scripts").exists():
        return here
    if (here.parent / "scripts").exists():
        return here.parent
    raise RuntimeError("Could not locate the repository root from the current working directory.")


def notebook_artifact_dir(name: str) -> Path:
    path = repo_root() / "notebooks" / "artifacts" / name
    path.mkdir(parents=True, exist_ok=True)
    return path


def configure_runtime(cache_subdir: str = "cache") -> dict[str, object]:
    root = repo_root()
    cache_root = root / "notebooks" / "artifacts" / cache_subdir
    cache_root.mkdir(parents=True, exist_ok=True)
    (cache_root / "mplconfig").mkdir(parents=True, exist_ok=True)
    os.environ["XDG_CACHE_HOME"] = str(cache_root.resolve())
    os.environ["MPLCONFIGDIR"] = str((cache_root / "mplconfig").resolve())

    import matplotlib.pyplot as plt
    import pennylane as qml

    plt.rcParams["figure.dpi"] = 1000
    plt.rcParams["savefig.dpi"] = 1000
    plt.rcParams["figure.figsize"] = (3, 2.5)
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.25
    plt.rcParams["axes.spines.top"] = False
    plt.rcParams["axes.spines.right"] = False
    pd.set_option("display.precision", 6)
    torch.set_num_threads(1)
    return {"plt": plt, "qml": qml}


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def accuracy(y_true, y_pred) -> float:
    yt = np.asarray(y_true)
    yp = np.asarray(y_pred)
    return 100.0 * float(np.mean(yt == yp))


class Standardizer:
    def fit(self, x: np.ndarray):
        flat = x.reshape(x.shape[0], -1).astype(np.float64)
        self.mu = flat.mean(axis=0)
        self.sig = flat.std(axis=0)
        self.sig[self.sig < 1e-8] = 1.0
        return self

    def transform(self, x: np.ndarray) -> np.ndarray:
        shape = x.shape
        flat = x.reshape(shape[0], -1).astype(np.float64)
        out = (flat - self.mu) / self.sig
        return out.reshape(shape).astype(np.float32)


def fit_ols(x_train: np.ndarray, y_train: np.ndarray) -> np.ndarray:
    a = np.hstack([x_train.astype(np.float64), np.ones((len(x_train), 1), dtype=np.float64)])
    return np.linalg.lstsq(a, y_train.astype(np.float64), rcond=None)[0].astype(np.float32)


def predict_ols(weights: np.ndarray, x: np.ndarray) -> np.ndarray:
    a = np.hstack([x.astype(np.float64), np.ones((len(x), 1), dtype=np.float64)])
    return (a @ weights.astype(np.float64)).astype(np.float32)


def fit_ridge_cv_binary(
    x_train: np.ndarray,
    y_train: np.ndarray,
    alphas: Iterable[float],
    val_frac: float = 0.2,
):
    split = max(1, int(len(x_train) * (1.0 - val_frac)))
    x_fit = x_train[:split].astype(np.float64)
    y_fit = y_train[:split].astype(np.float64)
    x_val = x_train[split:].astype(np.float64)
    y_val = y_train[split:]

    mu_x = x_fit.mean(axis=0)
    xc = x_fit - mu_x
    best_alpha = None
    best_acc = -np.inf
    for alpha in alphas:
        lam = np.linalg.solve(xc @ xc.T + alpha * np.eye(len(x_fit)), y_fit)
        w = xc.T @ lam
        b = float(y_fit.mean() - mu_x @ w)
        pred = ((x_val @ w + b) >= 0.5).astype(np.int64)
        acc = accuracy(y_val, pred)
        if acc > best_acc:
            best_acc = acc
            best_alpha = float(alpha)

    mu_x = x_train.astype(np.float64).mean(axis=0)
    xc = x_train.astype(np.float64) - mu_x
    lam = np.linalg.solve(xc @ xc.T + best_alpha * np.eye(len(x_train)), y_train.astype(np.float64))
    w = xc.T @ lam
    b = float(y_train.mean() - mu_x @ w)
    return w.astype(np.float32), b, best_alpha


def predict_binary_linear(w: np.ndarray, b: float, x: np.ndarray) -> np.ndarray:
    return ((x.astype(np.float64) @ w.astype(np.float64) + b) >= 0.5).astype(np.int64)


class BinaryMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int = 1):
        super().__init__()
        layers = [
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        out = self.net(x)
        if out.shape[-1] == 1:
            return out.squeeze(-1)
        return out


class MLQRCBinary(nn.Module):
    def __init__(self, t_dim: int, n_dim: int, f_dim: int, d_t: int, d_n: int, d_f: int):
        super().__init__()
        self.u_t = nn.Parameter(torch.randn(d_t, t_dim) * 0.05)
        self.u_n = nn.Parameter(torch.randn(d_n, n_dim) * 0.05)
        self.u_f = nn.Parameter(torch.randn(d_f, f_dim) * 0.05)
        self.head = nn.Linear(d_t * d_n * d_f, 1)

    def forward(self, x):
        y = torch.einsum("btnf,dt->bdnf", x, self.u_t)
        y = torch.einsum("bdnf,in->bdif", y, self.u_n)
        y = torch.einsum("bdif,jf->bdij", y, self.u_f)
        return self.head(y.reshape(y.shape[0], -1)).squeeze(-1)


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_binary_classifier(
    model: nn.Module,
    x_train: np.ndarray,
    y_train: np.ndarray,
    *,
    epochs: int,
    batch_size: int,
    lr: float,
    weight_decay: float,
    val_frac: float | None = None,
    patience: int = 25,
    num_classes: int = 1,
) -> nn.Module:
    model = model.to(DEVICE)

    if val_frac and len(x_train) > 20:
        split = max(1, int(len(x_train) * (1.0 - val_frac)))
        x_fit = x_train[:split]
        y_fit = y_train[:split]
        x_val = torch.tensor(x_train[split:], dtype=torch.float32, device=DEVICE)
        y_val = torch.tensor(y_train[split:], dtype=torch.long if num_classes > 1 else torch.float32, device=DEVICE)
    else:
        x_fit = x_train
        y_fit = y_train
        x_val = None
        y_val = None

    ds = TensorDataset(
        torch.tensor(x_fit, dtype=torch.float32),
        torch.tensor(y_fit, dtype=torch.long if num_classes > 1 else torch.float32),
    )
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss() if num_classes > 1 else nn.BCEWithLogitsLoss()

    best_state = None
    best_val = np.inf
    stale = 0
    for _ in range(epochs):
        model.train()
        for xb, yb in dl:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb if num_classes > 1 else yb)
            loss.backward()
            opt.step()

        if x_val is None:
            continue
        model.eval()
        with torch.no_grad():
            out = model(x_val)
            val = float(loss_fn(out, y_val if num_classes > 1 else y_val).item())
        if val + 1e-8 < best_val:
            best_val = val
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_binary_classifier(model: nn.Module, x: np.ndarray, num_classes: int = 1) -> np.ndarray:
    model.eval()
    out = model(torch.tensor(x, dtype=torch.float32, device=DEVICE))
    if num_classes > 1:
        return out.argmax(dim=1).cpu().numpy().astype(np.int64)
    return (out >= 0).long().cpu().numpy().astype(np.int64)


def unfold(x: np.ndarray, mode: int) -> np.ndarray:
    return np.reshape(np.moveaxis(x, mode, 0), (x.shape[mode], -1))


def truncated_left_singular_vectors(mat: np.ndarray, rank: int) -> np.ndarray:
    u, _, _ = np.linalg.svd(mat, full_matrices=False)
    return u[:, : min(rank, u.shape[1])].astype(np.float32)


def apply_mode_product(x: np.ndarray, factor: np.ndarray, mode: int) -> np.ndarray:
    moved = np.moveaxis(x, mode, 0)
    flat = moved.reshape(moved.shape[0], -1)
    projected = factor.T @ flat
    shape = (factor.shape[1],) + moved.shape[1:]
    return np.moveaxis(projected.reshape(shape), 0, mode)


def tensor_ridge_project_train(x_train: np.ndarray, target_ranks: tuple[int, int, int]):
    t_rank = min(target_ranks[0], x_train.shape[1])
    n_rank = min(target_ranks[1], x_train.shape[2])
    f_rank = min(target_ranks[2], x_train.shape[3])
    u_t = truncated_left_singular_vectors(unfold(x_train, 1), t_rank)
    u_n = truncated_left_singular_vectors(unfold(x_train, 2), n_rank)
    u_f = truncated_left_singular_vectors(unfold(x_train, 3), f_rank)
    return u_t, u_n, u_f


def tensor_project(x: np.ndarray, u_t: np.ndarray, u_n: np.ndarray, u_f: np.ndarray) -> np.ndarray:
    out = apply_mode_product(x, u_t, 1)
    out = apply_mode_product(out, u_n, 2)
    out = apply_mode_product(out, u_f, 3)
    return out


def build_tfim_hamiltonian(n_qubits: int, g: float, j: float = 1.0, periodic: bool = True) -> np.ndarray:
    dim = 2**n_qubits
    h = np.zeros((dim, dim), dtype=np.float64)
    for i in range(n_qubits):
        nxt = (i + 1) % n_qubits if periodic else (None if i == n_qubits - 1 else i + 1)
        if nxt is None:
            continue
        for s in range(dim):
            bi = (s >> (n_qubits - 1 - i)) & 1
            bj = (s >> (n_qubits - 1 - nxt)) & 1
            h[s, s] += -j * (1 - 2 * bi) * (1 - 2 * bj)
    for i in range(n_qubits):
        flip_mask = 1 << (n_qubits - 1 - i)
        for s in range(dim):
            h[s, s ^ flip_mask] += -g
    return h


def tfim_ground_state(n_qubits: int, g: float) -> np.ndarray:
    evals, evecs = np.linalg.eigh(build_tfim_hamiltonian(n_qubits, g))
    return evecs[:, 0].astype(np.complex128)


def sample_z_sequences(
    ground_state: np.ndarray,
    n_qubits: int,
    n_sequences: int,
    t_steps: int,
    rng: np.random.Generator,
) -> np.ndarray:
    probs = np.abs(ground_state) ** 2
    probs /= probs.sum()
    indices = rng.choice(2**n_qubits, size=n_sequences * t_steps, p=probs)
    shifts = np.arange(n_qubits - 1, -1, -1)
    bits = ((indices[:, None] >> shifts) & 1).astype(np.float32)
    return bits.reshape(n_sequences, t_steps, n_qubits)


def build_reservoir_unitary(qml, n_qubits: int, n_layers: int, reservoir_seed: int):
    rng = np.random.default_rng(reservoir_seed)
    weights = rng.uniform(0.0, 2.0 * np.pi, size=(n_layers, n_qubits, 3)).astype(np.float32)
    return torch.tensor(
        qml.matrix(qml.StronglyEntanglingLayers(weights, wires=range(n_qubits)), wire_order=range(n_qubits)),
        dtype=torch.complex64,
    )


def observable_matrices(qml, n_qubits: int):
    def op_mat(op):
        return torch.tensor(qml.matrix(op, wire_order=range(n_qubits)), dtype=torch.complex64)

    out = [op_mat(qml.PauliZ(i)) for i in range(n_qubits)]
    for i in range(n_qubits):
        for j in range(i + 1, n_qubits):
            out.append(op_mat(qml.PauliZ(i) @ qml.PauliZ(j)))
    return out


def batched_ry(states: torch.Tensor, angles: torch.Tensor, wire: int, n_qubits: int) -> torch.Tensor:
    c = torch.cos(angles / 2).to(torch.complex64)
    s = torch.sin(angles / 2).to(torch.complex64)
    gate = torch.stack(
        [torch.stack([c, -s], dim=-1), torch.stack([s, c], dim=-1)],
        dim=-2,
    )
    shaped = states.reshape(states.shape[0], *([2] * n_qubits))
    shaped = torch.movedim(shaped, 1 + wire, -1)
    shaped = torch.einsum("bij,b...j->b...i", gate, shaped)
    shaped = torch.movedim(shaped, -1, 1 + wire)
    return shaped.reshape(states.shape[0], -1)


def flatten_features(x: np.ndarray) -> np.ndarray:
    return x.reshape(x.shape[0], -1).astype(np.float32)


def mlqrc_param_count(t_steps: int, n_qubits: int, f_dim: int, d: int) -> int:
    return d * t_steps + d * n_qubits + d * f_dim + d**3 + 1


def fit_pca_projection(x_train_flat: np.ndarray, target_dim: int, seed: int = 0) -> tuple[PCA, int]:
    effective_dim = min(target_dim, x_train_flat.shape[0] - 1, x_train_flat.shape[1])
    pca = PCA(
        n_components=effective_dim,
        svd_solver="randomized",
        random_state=seed,
    )
    pca.fit(x_train_flat.astype(np.float64))
    return pca, int(effective_dim)


def build_phase_bar(main_df: pd.DataFrame, artifact_dir: Path, plt) -> None:
    color_map = {
        "Linear": "#c7e9c0",
        "Ridge": "#74c476",
        "MLP": "#bcbddc",
        "MLP-S": "#9e9ac8",
        "Ridge-PCA-d3": "#9ecae1",
        "Tensor-Ridge": "#fd8d3c",
        "ML-QRC (ours)": "#005a32",
    }
    plot_df = main_df.copy()
    fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.6), sharey=True)
    y_pos = np.arange(len(plot_df))
    for ax, regime, title in zip(
        axes,
        ["full", "half_data"],
        ["Full data (2240)", "50% data (1120)"],
    ):
        means = plot_df[f"{regime}_mean"].to_numpy(dtype=np.float64)
        stds = plot_df[f"{regime}_std"].to_numpy(dtype=np.float64)
        colors = [color_map[label] for label in plot_df["Method"]]
        ax.barh(y_pos, means, xerr=stds, color=colors, capsize=3)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(plot_df["Method"], fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel("Accuracy (%)")
        ax.set_title(title)
        ax.set_xlim(50, 75)
        ax.grid(True, axis="x", alpha=0.25)
        ax.grid(False, axis="y")
    plt.tight_layout()
    fig.savefig(artifact_dir / "fig_phase_bar.pdf", bbox_inches="tight")
    fig.savefig(artifact_dir / "fig_phase_bar.png", bbox_inches="tight")
    plt.close(fig)


def build_d_sweep_plot(
    d_df: pd.DataFrame,
    ridge_full: float,
    ridge_half: float,
    artifact_dir: Path,
    plt,
) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.4))
    mlqrc_df = d_df[d_df["Config"].isin(["ML-QRC-S", "ML-QRC-M", "ML-QRC-L", "ML-QRC-XL"])].copy()
    mlqrc_params = mlqrc_df["Params"].to_numpy(dtype=np.float64)
    mlqrc_full = mlqrc_df["full_mean"].to_numpy(dtype=np.float64)
    mlqrc_full_std = mlqrc_df["full_std"].to_numpy(dtype=np.float64)
    mlqrc_half = mlqrc_df["half_data_mean"].to_numpy(dtype=np.float64)
    mlqrc_half_std = mlqrc_df["half_data_std"].to_numpy(dtype=np.float64)
    ref_mlp = d_df[d_df["Config"] == "MLP (ref.)"].iloc[0]
    ref_mlps = d_df[d_df["Config"] == "MLP-S (ref.)"].iloc[0]

    for ax, regime, means, stds, title in [
        (axes[0], "full", mlqrc_full, mlqrc_full_std, "Full data (2240)"),
        (axes[1], "half_data", mlqrc_half, mlqrc_half_std, "50% data (1120)"),
    ]:
        ax.errorbar(
            mlqrc_params,
            means,
            yerr=stds,
            fmt="o-",
            color="#005a32",
            capsize=3,
            linewidth=1.5,
            markersize=5,
            label="ML-QRC sweep",
        )
        for label, x_val, y_val in zip(["d=4", "d=8", "d=16", "d=32"], mlqrc_params, means):
            ax.annotate(label, (x_val, y_val), xytext=(4, 4), textcoords="offset points", fontsize=7)
        ax.scatter(ref_mlp["Params"], ref_mlp["full_mean" if regime == "full" else "half_data_mean"], color="#756bb1", s=34, label="MLP")
        ax.scatter(ref_mlps["Params"], ref_mlps["full_mean" if regime == "full" else "half_data_mean"], color="#9e9ac8", s=34, label="MLP-S")
        ax.axhline(ridge_full if regime == "full" else ridge_half, color="#74c476", linestyle="--", linewidth=1.1, label="Ridge")
        ax.set_xscale("log")
        ax.set_xlabel("Trainable parameters")
        ax.set_ylabel("Accuracy (%)")
        ax.set_title(title)
        ax.set_ylim(55, 72)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.08))
    plt.tight_layout()
    fig.savefig(artifact_dir / "fig_phase_d_sweep.pdf", bbox_inches="tight")
    fig.savefig(artifact_dir / "fig_phase_d_sweep.png", bbox_inches="tight")
    plt.close(fig)


def sample_task2_subset(
    *,
    regime: str,
    seed: int,
    full_train: int,
    half_train: int,
    coupling_train_positions: list[np.ndarray],
) -> np.ndarray:
    if regime == "full":
        return np.arange(full_train, dtype=np.int64)

    rng = np.random.default_rng(seed)
    keep_per_coupling = half_train // len(coupling_train_positions)
    subset = np.concatenate(
        [
            rng.choice(coupling_pos, size=keep_per_coupling, replace=False)
            for coupling_pos in coupling_train_positions
        ]
    )
    rng.shuffle(subset)
    return subset.astype(np.int64)


def build_tfim_qrc_features(
    sequences: np.ndarray,
    *,
    qml,
    n_qubits: int,
    n_layers: int,
    t_steps: int,
    reservoir_seed: int,
    p_dep: float,
    cache_path: Path,
    batch_size: int = 256,
) -> np.ndarray:
    f_local = n_qubits
    f_pair = n_qubits * (n_qubits - 1) // 2
    f_dim = f_local + f_pair
    expected_shape = (len(sequences), t_steps, n_qubits, f_dim)
    if cache_path.exists():
        cached = np.load(cache_path)
        if cached.shape == expected_shape:
            return cached.astype(np.float32)

    reservoir = build_reservoir_unitary(qml, n_qubits, n_layers, reservoir_seed).to(DEVICE)
    observables = [obs.to(DEVICE) for obs in observable_matrices(qml, n_qubits)]
    zero_state = torch.zeros(2**n_qubits, dtype=torch.complex64, device=DEVICE)
    zero_state[0] = 1.0
    pi_val = torch.tensor(float(np.pi), dtype=torch.float32, device=DEVICE)
    qi = torch.arange(n_qubits, device=DEVICE)
    scale_local = float(1.0 - 4.0 * p_dep / 3.0)
    scale_pair = float(scale_local**2)

    seq_tensor = torch.tensor(sequences, dtype=torch.float32, device=DEVICE)
    chunks = []
    for start in range(0, len(seq_tensor), batch_size):
        batch = seq_tensor[start : start + batch_size]
        bs = batch.shape[0]
        states = zero_state.unsqueeze(0).expand(bs, -1).clone()
        feat = torch.zeros(bs, t_steps, n_qubits, f_dim, dtype=torch.float32, device=DEVICE)
        for t_idx in range(t_steps):
            angles = pi_val * batch[:, t_idx, :]
            for wire in range(n_qubits):
                states = batched_ry(states, angles[:, wire], wire, n_qubits)
            states = states @ reservoir.T
            evs = torch.stack(
                [torch.real(torch.sum(states.conj() * (states @ obs.T), dim=1)) for obs in observables],
                dim=1,
            )
            evs[:, :f_local] *= scale_local
            evs[:, f_local:] *= scale_pair
            feat[:, t_idx, qi, qi] = evs[:, :f_local]
            pair = 0
            for i in range(n_qubits):
                for j in range(i + 1, n_qubits):
                    col = f_local + pair
                    feat[:, t_idx, i, col] = evs[:, col]
                    feat[:, t_idx, j, col] = evs[:, col]
                    pair += 1
        chunks.append(feat.cpu().numpy())
    out = np.concatenate(chunks, axis=0).astype(np.float32)
    np.save(cache_path, out)
    return out


## Configuration

The cell below configures the experiment and resolves the artifact directory used by the notebook.


In [ ]:
runtime = configure_runtime()
plt = runtime["plt"]
qml = runtime["qml"]
ROOT = repo_root()
ARTIFACT_DIR = notebook_artifact_dir("task3_tfim")

CONFIG = {
    "eval_seeds": list(range(10)),
    "reservoir_seed": 42,
    "data_seed": 42,
    "n_g_values": 16,
    "g_min": 0.2,
    "g_max": 1.8,
    "n_sequences": 200,
    "train_frac": 0.70,
    "n_qubits": 6,
    "n_layers": 1,
    "t_steps": 50,
    "p_dep": 0.01,
    "batch_size_features": 256,
    "epochs": 300,
    "batch_size": 32,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "mlp_hidden": 64,
    "mlp_s_pca_dim": 32,
    "mlp_s_hidden": 32,
    "ridge_pca_dim": 512,
    "tensor_ranks": (8, 6, 8),
    "ridge_alphas": (1e-4, 1e-3, 1e-2, 0.1, 1.0, 10.0, 100.0, 1e3, 1e4, 1e5),
}

T_STEPS = CONFIG["t_steps"]
N_QUBITS = CONFIG["n_qubits"]
F_DIM = N_QUBITS + N_QUBITS * (N_QUBITS - 1) // 2
FLAT_DIM = T_STEPS * N_QUBITS * F_DIM
FULL_TRAIN = int(round(CONFIG["n_g_values"] * CONFIG["n_sequences"] * CONFIG["train_frac"]))
HALF_TRAIN = FULL_TRAIN // 2

print(f"Repo root: {ROOT}")
print(f"Artifacts: {ARTIFACT_DIR}")
print(f"Torch device: {DEVICE}")
print(pd.Series(CONFIG, name="value"))


## Data Generation And Split

This follows the manuscript Task 2 protocol and keeps the low-data regime balanced across all TFIM couplings.


In [ ]:
g_values = np.linspace(CONFIG["g_min"], CONFIG["g_max"], CONFIG["n_g_values"])
g_labels = np.array([0 if g < 1 else 1 for g in g_values], dtype=np.int64)

rng_data = np.random.default_rng(CONFIG["data_seed"])
sequences = []
labels = []
coupling_ids = []

for gi, (g, label) in enumerate(zip(g_values, g_labels)):
    ground_state = tfim_ground_state(N_QUBITS, float(g))
    seq = sample_z_sequences(ground_state, N_QUBITS, CONFIG["n_sequences"], T_STEPS, rng_data)
    sequences.append(seq)
    labels.extend([label] * CONFIG["n_sequences"])
    coupling_ids.extend([gi] * CONFIG["n_sequences"])

sequences = np.concatenate(sequences, axis=0).astype(np.float32)
labels = np.asarray(labels, dtype=np.int64)
coupling_ids = np.asarray(coupling_ids, dtype=np.int64)

rng_split = np.random.default_rng(CONFIG["data_seed"] + 1)
train_idx = []
test_idx = []
per_block_train = round(CONFIG["n_sequences"] * CONFIG["train_frac"])
for gi in range(CONFIG["n_g_values"]):
    block = np.arange(gi * CONFIG["n_sequences"], (gi + 1) * CONFIG["n_sequences"])
    rng_split.shuffle(block)
    train_idx.extend(block[:per_block_train])
    test_idx.extend(block[per_block_train:])

train_idx = np.asarray(train_idx, dtype=np.int64)
test_idx = np.asarray(test_idx, dtype=np.int64)
y_train = labels[train_idx]
y_test = labels[test_idx]
train_couplings = coupling_ids[train_idx]

assert len(train_idx) == FULL_TRAIN
assert len(test_idx) == len(labels) - FULL_TRAIN
assert FULL_TRAIN % CONFIG["n_g_values"] == 0
assert HALF_TRAIN % CONFIG["n_g_values"] == 0

coupling_train_positions = [np.where(train_couplings == gi)[0] for gi in range(CONFIG["n_g_values"])]
train_per_coupling = FULL_TRAIN // CONFIG["n_g_values"]
half_per_coupling = HALF_TRAIN // CONFIG["n_g_values"]
assert all(len(pos) == train_per_coupling for pos in coupling_train_positions)

print(f"TFIM samples: {sequences.shape}")
print(f"Train/Test sizes: {len(train_idx)} / {len(test_idx)}")
print(
    "Low-data regime: stratified per coupling "
    f"({half_per_coupling} train sequences from each of {CONFIG['n_g_values']} couplings)."
)


In [ ]:
ground_fig, axes = plt.subplots(1, 2, figsize=(8.0, 2.6))
for ax, g, title in zip(
    axes,
    [0.5, 1.5],
    ["FM ground state (g=0.5)", "PM ground state (g=1.5)"],
):
    gs = tfim_ground_state(N_QUBITS, g)
    ax.bar(np.arange(2**N_QUBITS), np.abs(gs) ** 2, color="#4292c6", width=0.8)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Basis state index")
    ax.set_ylabel("Probability")

plt.tight_layout()
display(ground_fig)
ground_fig.savefig(ARTIFACT_DIR / "tfim_ground_states.pdf", bbox_inches="tight")
ground_fig.savefig(ARTIFACT_DIR / "tfim_ground_states.png", bbox_inches="tight")
plt.close(ground_fig)


## QRC Feature Bank

The feature tensor is cached under the artifact directory so repeated notebook runs do not rebuild it unnecessarily.


In [ ]:
qrc_cache = ARTIFACT_DIR / (
    f"qrc_feats_tfim_rseed{CONFIG['reservoir_seed']}_p{CONFIG['p_dep']:.2f}_q{N_QUBITS}_T{T_STEPS}.npy"
)
qrc_all = build_tfim_qrc_features(
    sequences,
    qml=qml,
    n_qubits=N_QUBITS,
    n_layers=CONFIG["n_layers"],
    t_steps=T_STEPS,
    reservoir_seed=CONFIG["reservoir_seed"],
    p_dep=CONFIG["p_dep"],
    cache_path=qrc_cache,
    batch_size=CONFIG["batch_size_features"],
)
train_qrc_all = qrc_all[train_idx]
test_qrc = qrc_all[test_idx]

print(f"QRC tensor bank: {qrc_all.shape}")
print(f"Flat dimension: {FLAT_DIM}")
print(f"Cached at: {qrc_cache}")


## Evaluation

This cell recomputes the baselines and ML-QRC sweep from the cached QRC tensor bank.


In [ ]:
method_order = [
    "Linear",
    "Ridge",
    "MLP",
    "MLP-S",
    "Ridge-PCA-d3",
    "Tensor-Ridge",
    "ML-QRC-S",
    "ML-QRC-M",
    "ML-QRC-L",
    "ML-QRC-XL",
]
regimes = {"full": FULL_TRAIN, "half_data": HALF_TRAIN}
results = {regime: {method: [] for method in method_order} for regime in regimes}
detail_rows = []

t0 = time.time()
total_jobs = len(CONFIG["eval_seeds"]) * len(regimes)
job_done = 0

for seed in CONFIG["eval_seeds"]:
    for regime, n_train in regimes.items():
        subset = sample_task2_subset(
            regime=regime,
            seed=seed,
            full_train=FULL_TRAIN,
            half_train=HALF_TRAIN,
            coupling_train_positions=coupling_train_positions,
        )

        train_qrc = train_qrc_all[subset]
        train_y = y_train[subset]
        scaler = Standardizer().fit(train_qrc)
        x_train_t = scaler.transform(train_qrc)
        x_test_t = scaler.transform(test_qrc)
        x_train_f = flatten_features(x_train_t)
        x_test_f = flatten_features(x_test_t)

        w = fit_ols(x_train_f, train_y.astype(np.float32))
        pred = (predict_ols(w, x_test_f) >= 0.5).astype(np.int64)
        score = accuracy(y_test, pred)
        results[regime]["Linear"].append(score)
        detail_rows.append({"seed": seed, "regime": regime, "method": "Linear", "score": score})

        w, b, alpha = fit_ridge_cv_binary(x_train_f, train_y, CONFIG["ridge_alphas"])
        pred = predict_binary_linear(w, b, x_test_f)
        score = accuracy(y_test, pred)
        results[regime]["Ridge"].append(score)
        detail_rows.append({"seed": seed, "regime": regime, "method": "Ridge", "score": score, "alpha": alpha})

        set_seed(seed)
        model = BinaryMLP(x_train_f.shape[1], CONFIG["mlp_hidden"], num_classes=1)
        model = train_binary_classifier(
            model,
            x_train_f,
            train_y,
            epochs=CONFIG["epochs"],
            batch_size=CONFIG["batch_size"],
            lr=CONFIG["lr"],
            weight_decay=CONFIG["weight_decay"],
            num_classes=1,
        )
        pred = predict_binary_classifier(model, x_test_f, num_classes=1)
        score = accuracy(y_test, pred)
        results[regime]["MLP"].append(score)
        detail_rows.append({"seed": seed, "regime": regime, "method": "MLP", "score": score})

        pca_small, pca_small_dim = fit_pca_projection(x_train_f, CONFIG["mlp_s_pca_dim"], seed=0)
        z_train_small = pca_small.transform(x_train_f.astype(np.float64)).astype(np.float32)
        z_test_small = pca_small.transform(x_test_f.astype(np.float64)).astype(np.float32)
        set_seed(seed)
        model = BinaryMLP(z_train_small.shape[1], CONFIG["mlp_s_hidden"], num_classes=2)
        model = train_binary_classifier(
            model,
            z_train_small,
            train_y,
            epochs=CONFIG["epochs"],
            batch_size=CONFIG["batch_size"],
            lr=CONFIG["lr"],
            weight_decay=CONFIG["weight_decay"],
            num_classes=2,
        )
        pred = predict_binary_classifier(model, z_test_small, num_classes=2)
        score = accuracy(y_test, pred)
        results[regime]["MLP-S"].append(score)
        detail_rows.append(
            {
                "seed": seed,
                "regime": regime,
                "method": "MLP-S",
                "score": score,
                "pca_dim": pca_small_dim,
            }
        )

        pca_large, pca_large_dim = fit_pca_projection(x_train_f, CONFIG["ridge_pca_dim"], seed=0)
        z_train_large = pca_large.transform(x_train_f.astype(np.float64)).astype(np.float32)
        z_test_large = pca_large.transform(x_test_f.astype(np.float64)).astype(np.float32)
        w, b, alpha = fit_ridge_cv_binary(z_train_large, train_y, CONFIG["ridge_alphas"])
        pred = predict_binary_linear(w, b, z_test_large)
        score = accuracy(y_test, pred)
        results[regime]["Ridge-PCA-d3"].append(score)
        detail_rows.append(
            {
                "seed": seed,
                "regime": regime,
                "method": "Ridge-PCA-d3",
                "score": score,
                "alpha": alpha,
                "pca_dim": pca_large_dim,
            }
        )

        u_t, u_n, u_f = tensor_ridge_project_train(x_train_t, CONFIG["tensor_ranks"])
        core_train = flatten_features(tensor_project(x_train_t, u_t, u_n, u_f))
        core_test = flatten_features(tensor_project(x_test_t, u_t, u_n, u_f))
        w, b, alpha = fit_ridge_cv_binary(core_train, train_y, CONFIG["ridge_alphas"])
        pred = predict_binary_linear(w, b, core_test)
        score = accuracy(y_test, pred)
        results[regime]["Tensor-Ridge"].append(score)
        detail_rows.append(
            {
                "seed": seed,
                "regime": regime,
                "method": "Tensor-Ridge",
                "score": score,
                "alpha": alpha,
                "tensor_ranks": tuple(int(v) for v in (u_t.shape[1], u_n.shape[1], u_f.shape[1])),
            }
        )

        for d, name in [(4, "ML-QRC-S"), (8, "ML-QRC-M"), (16, "ML-QRC-L"), (32, "ML-QRC-XL")]:
            set_seed(seed)
            model = MLQRCBinary(T_STEPS, N_QUBITS, F_DIM, d, d, d)
            model = train_binary_classifier(
                model,
                x_train_t,
                train_y,
                epochs=CONFIG["epochs"],
                batch_size=CONFIG["batch_size"],
                lr=CONFIG["lr"],
                weight_decay=CONFIG["weight_decay"],
                num_classes=1,
            )
            pred = predict_binary_classifier(model, x_test_t, num_classes=1)
            score = accuracy(y_test, pred)
            results[regime][name].append(score)
            detail_rows.append({"seed": seed, "regime": regime, "method": name, "score": score, "d": d})

        job_done += 1
        elapsed = (time.time() - t0) / 60.0
        print(
            f"[{job_done:02d}/{total_jobs:02d}] seed={seed} regime={regime:<9s} "
            f"Ridge={results[regime]['Ridge'][-1]:.1f}  "
            f"MLP-S={results[regime]['MLP-S'][-1]:.1f}  "
            f"Tensor-Ridge={results[regime]['Tensor-Ridge'][-1]:.1f}  "
            f"ML-QRC-M={results[regime]['ML-QRC-M'][-1]:.1f}  "
            f"[{elapsed:.1f} min]"
        )

detail_df = pd.DataFrame(detail_rows)
detail_df.to_csv(ARTIFACT_DIR / "task2_run_details.csv", index=False)
print(f"Saved: {ARTIFACT_DIR / 'task2_run_details.csv'}")


## Tables And Figures

This cell assembles the manuscript-style outputs from the run-level results.


In [ ]:
main_methods = [
    ("Linear", "Linear"),
    ("Ridge", "Ridge"),
    ("MLP", "MLP"),
    ("MLP-S", "MLP-S"),
    ("Ridge-PCA-d3", "Ridge-PCA-d3"),
    ("Tensor-Ridge", "Tensor-Ridge"),
    ("ML-QRC-M", "ML-QRC (ours)"),
]

main_rows = []
for raw_key, label in main_methods:
    row = {"Method": label, "raw_key": raw_key}
    for regime in regimes:
        values = np.asarray(results[regime][raw_key], dtype=np.float64)
        row[f"{regime}_mean"] = float(values.mean())
        row[f"{regime}_std"] = float(values.std())

    if raw_key in {"Linear", "Ridge"}:
        row["Params"] = FLAT_DIM + 1
    elif raw_key == "MLP":
        row["Params"] = count_params(BinaryMLP(FLAT_DIM, CONFIG["mlp_hidden"], num_classes=1))
    elif raw_key == "MLP-S":
        row["Params"] = count_params(BinaryMLP(CONFIG["mlp_s_pca_dim"], CONFIG["mlp_s_hidden"], num_classes=2))
    elif raw_key == "Ridge-PCA-d3":
        row["Params"] = min(CONFIG["ridge_pca_dim"], FULL_TRAIN - 1, FLAT_DIM) + 1
    elif raw_key == "Tensor-Ridge":
        row["Params"] = int(np.prod(CONFIG["tensor_ranks"])) + 1
    else:
        row["Params"] = count_params(MLQRCBinary(T_STEPS, N_QUBITS, F_DIM, 8, 8, 8))

    main_rows.append(row)

main_df = pd.DataFrame(main_rows)
display(main_df)

table_phase = ARTIFACT_DIR / "table_phase.csv"
table_alias = ARTIFACT_DIR / "table2_main.csv"
main_df.to_csv(table_phase, index=False, float_format="%.6f")
main_df.to_csv(table_alias, index=False, float_format="%.6f")

d_rows = [
    {
        "Config": "MLP (ref.)",
        "raw_key": "MLP",
        "d": "—",
        "Params": count_params(BinaryMLP(FLAT_DIM, CONFIG["mlp_hidden"], num_classes=1)),
        "full_mean": float(np.mean(results["full"]["MLP"])),
        "full_std": float(np.std(results["full"]["MLP"])),
        "half_data_mean": float(np.mean(results["half_data"]["MLP"])),
        "half_data_std": float(np.std(results["half_data"]["MLP"])),
    },
    {
        "Config": "MLP-S (ref.)",
        "raw_key": "MLP-S",
        "d": "—",
        "Params": count_params(BinaryMLP(CONFIG["mlp_s_pca_dim"], CONFIG["mlp_s_hidden"], num_classes=2)),
        "full_mean": float(np.mean(results["full"]["MLP-S"])),
        "full_std": float(np.std(results["full"]["MLP-S"])),
        "half_data_mean": float(np.mean(results["half_data"]["MLP-S"])),
        "half_data_std": float(np.std(results["half_data"]["MLP-S"])),
    },
]

for d, name in [(4, "ML-QRC-S"), (8, "ML-QRC-M"), (16, "ML-QRC-L"), (32, "ML-QRC-XL")]:
    d_rows.append(
        {
            "Config": name,
            "raw_key": name,
            "d": d,
            "Params": mlqrc_param_count(T_STEPS, N_QUBITS, F_DIM, d),
            "full_mean": float(np.mean(results["full"][name])),
            "full_std": float(np.std(results["full"][name])),
            "half_data_mean": float(np.mean(results["half_data"][name])),
            "half_data_std": float(np.std(results["half_data"][name])),
        }
    )

d_df = pd.DataFrame(d_rows)
display(d_df)

d_csv = ARTIFACT_DIR / "table10_d_sweep.csv"
d_alias = ARTIFACT_DIR / "phase_ablation.csv"
d_df.to_csv(d_csv, index=False, float_format="%.6f")
d_df.to_csv(d_alias, index=False, float_format="%.6f")

build_phase_bar(main_df, ARTIFACT_DIR, plt)
ridge_row = main_df.loc[main_df["Method"] == "Ridge"].iloc[0]
build_d_sweep_plot(
    d_df,
    ridge_full=float(ridge_row["full_mean"]),
    ridge_half=float(ridge_row["half_data_mean"]),
    artifact_dir=ARTIFACT_DIR,
    plt=plt,
)

print(f"Saved: {table_phase}")
print(f"Saved alias: {table_alias}")
print(f"Saved: {d_csv}")
print(f"Saved compatibility copy: {d_alias}")
print(f"Saved: {ARTIFACT_DIR / 'fig_phase_bar.pdf'}")
print(f"Saved: {ARTIFACT_DIR / 'fig_phase_d_sweep.pdf'}")


## Outputs

After execution, the notebook writes the Task 2 artifacts under `notebooks/artifacts/task3_tfim/`:

- `qrc_feats_tfim_rseed42_p0.01_q6_T50.npy`
- `table_phase.csv`
- `table2_main.csv`
- `table10_d_sweep.csv`
- `phase_ablation.csv`
- `fig_phase_bar.pdf`
- `fig_phase_d_sweep.pdf`
- `tfim_ground_states.pdf`
- `task2_run_details.csv`
